# Restvolumen je Projekt

Schritt 1 aus Spec Abschnitt 10: `/v4/projects` und `/v2/entrygroups` abfragen und
das Restvolumen je Projekt berechnen (Abschnitt 5.1) – als Zwischenschritt **vor**
der Monte-Carlo-Logik.

> **Datenlage:** `docs.clockodo.com` wird als JavaScript-Anwendung ausgeliefert und war
> nicht auslesbar. Envelope, Query-Parameter und Feldnamen beider Endpunkte sind
> deshalb am 24.08.2026 per `curl` gegen die echte Installation geprüft – die Details
> stehen als Kommentar an der jeweiligen Zelle und im Docstring von
> `umsatzprognose.extraktion`. Offen sind keine Strukturfragen mehr, sondern zwei
> fachliche Abgrenzungen, mit `ENTSCHEIDEN` markiert.

In [1]:
# Nur in Google Colab: Projekt installieren. Lokal passiert hier nichts,
# dort liefert `uv sync` die Umgebung.
try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import os

    from google.colab import userdata

    def secret(name):
        """Colab-Secret lesen und im Fehlerfall sagen, was zu tun ist."""
        try:
            return userdata.get(name)
        except Exception as fehler:
            raise RuntimeError(
                f"Colab-Secret '{name}' nicht nutzbar ({type(fehler).__name__}).\n"
                "Anlegen: linke Seitenleiste, Schluessel-Symbol -> 'Neues Secret'.\n"
                "Danach den Schalter 'Notebook-Zugriff' fuer dieses Notebook "
                "aktivieren - ohne ihn existiert das Secret, ist aber gesperrt.\n"
                "Details im README, Abschnitt 'Deployment (Google Colab)'."
            ) from fehler

    # Das Repository ist oeffentlich, deshalb braucht pip kein Token.
    # PAKET_REF auf einen Tag setzen, damit der Lauf reproduzierbar bleibt.
    os.environ["PAKET_REF"] = "main"

    # Die Abhaengigkeiten (httpx, pandas, python-dotenv) kommen als Requirements mit.
    !pip install --quiet "git+https://github.com/it-agile/umsatzprognose-clockodo.git@$PAKET_REF"

IN_COLAB

False

In [ ]:
import httpx
import pandas as pd

from umsatzprognose.config import BASE_URL, load_credentials
from umsatzprognose.extraktion import budgets_je_projekt, revenue_je_projekt
from umsatzprognose.restvolumen import restvolumen_je_projekt, summe_prognosewirksam

if IN_COLAB:
    # In Colab die Werte aus der Secrets-Verwaltung in die Umgebung heben,
    # dann `use_dotenv=False`. Keine .env in Colab anlegen.
    import os

    for key in (
        "CLOCKODO_API_USER",
        "CLOCKODO_API_KEY",
        "CLOCKODO_APP_NAME",
        "CLOCKODO_APP_EMAIL",
    ):
        os.environ[key] = secret(key)

creds = load_credentials(use_dotenv=not IN_COLAB)
print("Angemeldet als", creds.api_user)

In [ ]:
class ClockodoError(RuntimeError):
    """HTTP-Fehler samt Antwortkoerper.

    ``raise_for_status`` zeigt nur Status und URL. Clockodo begruendet einen 400
    aber im Koerper und benennt dort den beanstandeten Parameter - genau die
    Information, die hier gebraucht wird.
    """


def get(path: str, params: dict | None = None, **kwargs):
    """Ein GET gegen die Clockodo-API. Wirft bei HTTP-Fehlern.

    Parameter gehen als Dict oder als Schluesselwoerter rein. Das Dict ist noetig
    fuer Namen wie ``grouping[]``, die kein gueltiges Python-Schluesselwort sind.
    """
    alle = {**(params or {}), **kwargs}
    with httpx.Client(base_url=BASE_URL, headers=creds.headers(), timeout=30.0) as client:
        response = client.get(path, params=alle or None)
        if response.is_error:
            raise ClockodoError(
                f"{response.status_code} fuer {response.request.url}\n{response.text[:1000]}"
            )
        return response.json()


def zeige_struktur(name: str, payload) -> None:
    """Top-Level-Struktur ausgeben, um das Envelope zu identifizieren."""
    if isinstance(payload, dict):
        print(f"{name}: dict mit Keys {sorted(payload)}")
    elif isinstance(payload, list):
        print(f"{name}: Liste mit {len(payload)} Eintraegen")
        if payload:
            print(f"  erster Eintrag, Keys: {sorted(payload[0])}")
    else:
        print(f"{name}: {type(payload)}")

## Auftragsvolumen aus `/v4/projects`

In [ ]:
# /v4/projects liefert {"paging": {...}, "data": [...]}. Verifiziert am 24.08.2026:
# "items_per_page" setzt die Seitengroesse, "page" waehlt die Seite - mit
# items_per_page=3 antwortet die API mit count_pages=299, und page=2 liefert
# current_page=2 und andere IDs.
#
# Achtung bei weiteren Versuchen: unbekannte Query-Parameter werden still
# ignoriert, nicht abgelehnt ("count=3" und "limit=3" antworten mit 200 und den
# vollen 895 Projekten). Ein 200 belegt einen Parameternamen also nicht - dafuer
# muss das paging-Objekt der Antwort geprueft werden.
def get_paged(path, **params):
    """Alle Seiten eines v4-Endpunkts einsammeln."""
    seite, alle, paging = 1, [], {}
    while True:
        payload = get(path, page=seite, **params)
        alle.extend(payload["data"])
        paging = payload.get("paging") or {}
        if seite >= paging.get("count_pages", 1):
            return alle, paging
        seite += 1


projects, paging = get_paged("/v4/projects")
print(f"{len(projects)} Projekte geladen, paging: {paging}")

In [ ]:
# ENTSCHEIDEN: Von 895 Projekten sind nur 122 aktiv, der Rest ist abgeschlossen
# oder archiviert. Fuer eine Prognose kuenftiger Umsaetze zaehlen laufende
# Projekte. Die Spec sagt dazu nichts - die Abgrenzung ist eine Annahme und
# gehoert bestaetigt oder korrigiert.
NUR_AKTIVE = True

auszug = budgets_je_projekt(projects, nur_aktive=NUR_AKTIVE)
budgets = auszug.budgets

ohne = sum(v is None for v in budgets.values())
print(f"{len(budgets)} von {len(projects)} Projekten beruecksichtigt (NUR_AKTIVE={NUR_AKTIVE})")
print(f"davon {ohne} ohne verwertbares Euro-Gesamtbudget")

# Sonderformen des Budgets, die "budget.amount" seine Euro-Bedeutung nehmen. Bei
# den aktiven Projekten bisher leer; die Liste macht sichtbar, wenn das kippt.
if auszug.unbenutzbar:
    print(f"  Stundenbudget (monetary=false): {auszug.nicht_monetaer}")
    print(f"  Budget je Intervall:            {auszug.mit_intervall}")
    print(f"  Budget aus Teilprojekten:       {auszug.aus_teilprojekten}")

# Zwei aktive Projekte tragen completed=true - abgeschlossen, aber nicht
# archiviert. Ihr Restbudget geht hier in die Prognose ein. ENTSCHEIDEN, ob das
# gewollt ist; die Spec kennt das Feld nicht.
abgeschlossen = [p for p in projects if p.get("active") and p.get("completed")]
print(f"aktiv und trotzdem completed: {len(abgeschlossen)}")

## Verbrauchtes Volumen aus `/v2/entrygroups`

Gruppierung nach Projekt, Zeitraum weit genug, um die gesamte Projekthistorie zu
erfassen – `revenue_kumuliert` in Abschnitt 5.1 ist der Gesamtverbrauch, nicht der
eines Monats.

In [ ]:
# Verifiziert am 24.08.2026 an echten Antworten von /v2/entrygroups:
# - grouping ist ein Array-Parameter. "grouping": "projects_id" antwortet mit
#   400 {"error":{"message":"Array expected.","fields":["grouping"]}}.
# - Gueltiger Gruppierungswert ist "projects_id", nicht "projects" - letzteres
#   gibt "Unknown group option".
# - grouping und time_since sind Pflicht ("Missing data: ...").
# - Zeitgrenzen brauchen die volle ISO-Form mit Uhrzeit; ein reines Datum gibt
#   "Wrong format".
# Der untere Rand 2020 schneidet nichts ab: mit time_since=2010-01-01 kommen
# dieselben 870 Gruppen und dieselbe Umsatzsumme zurueck.
ZEITRAUM_VON = "2020-01-01T00:00:00Z"
ZEITRAUM_BIS = "2026-12-31T23:59:59Z"

ENTRYGROUPS_PARAMS = {
    "time_since": ZEITRAUM_VON,
    "time_until": ZEITRAUM_BIS,
    "grouping[]": "projects_id",
}

entrygroups_raw = get("/v2/entrygroups", ENTRYGROUPS_PARAMS)
zeige_struktur("entrygroups_raw", entrygroups_raw)

In [ ]:
# Envelope-Key "groups" ist verifiziert (24.08.2026), ebenso die Feldnamen:
# "group" (Projekt-ID, als String), "revenue" in Euro, "duration" in Sekunden.
groups = entrygroups_raw["groups"]
revenue_kumuliert, ohne_projekt = revenue_je_projekt(groups)

print(f"{len(groups)} Gruppen geladen, Verbrauch fuer {len(revenue_kumuliert)} Projekte")
print(
    f"Gruppen ohne Projektbezug (group == 0): {len(ohne_projekt)}, "
    f"Umsatz {sum(float(g.get('revenue') or 0) for g in ohne_projekt):,.2f} EUR"
)

# "hourly_rate" ist genau dann gesetzt, wenn
# hourly_rate_is_equal_and_has_no_lumpsums true ist - in dieser Installation bei
# 92 von 870 Gruppen, und dort ueberwiegend 0. Fuer die 778 Projekte mit
# gemischten Saetzen oder Pauschalleistungen ist das Feld null, taugt also nicht
# als effektiver Stundensatz. Der muss aus revenue und duration abgeleitet
# werden; die Definition steht in Spec v0.3, die nicht vorliegt.
gemischt = sum(1 for g in groups if not g["hourly_rate_is_equal_and_has_no_lumpsums"])
print(f"Gruppen mit gemischten Saetzen oder Pauschalleistungen: {gemischt} von {len(groups)}")

## Restvolumen (Spec 5.1)

`roh` ist `budget.amount - revenue_kumuliert` und kann negativ sein, weil
`budget.hard` false ist. `prognosewirksam` kappt bei 0 – nur dieser Teil kann noch
abgerufen werden und geht in die Simulation ein.

Laut Spec 5.1 (seit v0.5) kann eine Budgetüberschreitung **nur historisch**
entstehen; die Prognose überschreitet das Budget nicht. Für Projekte mit
`ueberschritten == True` wird deshalb kein zukünftiger Umsatz prognostiziert.

In [ ]:
restvolumina, ohne_budget = restvolumen_je_projekt(budgets, revenue_kumuliert)

df = pd.DataFrame(
    [
        {
            "projects_id": r.projects_id,
            "budget": r.budget,
            "revenue_kumuliert": r.revenue_kumuliert,
            "restvolumen_roh": r.roh,
            "prognosewirksam": r.prognosewirksam,
            "ueberschritten": r.ueberschritten,
        }
        for r in restvolumina
    ]
).sort_values("prognosewirksam", ascending=False)

print(f"Prognosewirksames Restvolumen gesamt: {summe_prognosewirksam(restvolumina):,.2f} EUR")
print(f"Projekte mit Budgetueberschreitung:   {int(df['ueberschritten'].sum())}")
print(f"Projekte ohne Budget (ausgeschlossen): {len(ohne_budget)}, z. B. {ohne_budget[:5]}")

df.head(30)

## Offen

**ENTSCHEIDEN – 78 der 122 aktiven Projekte haben kein Budget** und fallen damit aus
der Prognose. Ein Blick auf die Namen zeigt, was das überwiegend ist: Schulungs- und
Ausbildungsprodukte (`A-CSM`, `A-CSPO`, `ACC`, `Agile Change Management`, …), also
Katalogpositionen ohne beauftragtes Volumen. Genau das rechnet die Spec dem
Kurzfristgeschäft zu und schließt es aus dem MVP aus. Zu prüfen bleibt, ob unter den
78 auch echte Bestandsprojekte stecken, bei denen nur das Budget fehlt – dann ist es
ein Pflegethema, kein Modellthema.

**ENTSCHEIDEN – zwei aktive Projekte sind `completed`,** eines davon mit 12.424 EUR
offenem Budget. Sie gehen derzeit in die Prognose ein, obwohl sie fachlich beendet
sein dürften. Das Feld kennt die Spec nicht.

**Effektiver Stundensatz:** `hourly_rate` aus `/v2/entrygroups` taugt dafür nicht – es
ist genau dann gesetzt, wenn das Projekt einen einheitlichen Satz und keine
Pauschalleistungen hat, also bei 92 von 870 Gruppen, und dort meist 0. Für die
restlichen 778 ist es `null`. Der Satz muss aus `revenue` und `duration` abgeleitet
werden; die Definition steht in Spec v0.3, die nicht im Repository liegt. Damit fehlt
auch die Normalisierung von Pauschalleistungen (5.1) – 8 Gruppen haben Umsatz ohne
jede erfasste Zeit.

**Nächster Schritt** laut Abschnitt 10: Abrufquoten-Verteilungen je Referenzklasse aus
der `entrygroups`-Historie schätzen.